<a href="https://colab.research.google.com/github/bansal19/exa-hack/blob/main/Llava_Logit_Lens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install nnsight

!pip install nnsight
from IPython.display import clear_output

clear_output()

In [ ]:
!pip install qwen-vl-utils
!pip install git+https://github.com/huggingface/transformers

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-3j82cvfc
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-3j82cvfc
  Resolved https://github.com/huggingface/transformers to commit 816f4424964c1a1631e303b663fc3d68f731e923
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch


# Do a forward pass with the Qwen model until the embedding layer
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct", torch_dtype="auto", device_map="auto"
)




/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

In [ ]:
tokenizer=AutoTokenizer.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

In [ ]:
processor()

ValueError: text input must be of type `str` (single example), `List[str]` (batch or single pretokenized example) or `List[List[str]]` (batch of pretokenized examples).

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
import torch

model = Qwen2VLForConditionalGeneration.from_pretrained("Qwen/Qwen2-VL-2B-Instruct", device_map="auto")
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
from PIL import Image
import requests

url = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
image = Image.open(requests.get(url, stream=True).raw)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# inputs = processor(images=image, return_tensors="pt").to(model.device)
inputs = processor.image_processor(images=image, return_tensors="pt").to(model.device)
# vision_tower = model.get_vision_tower()
# with torch.no_grad():
#     outputs = model(**inputs, output_hidden_states=True)
#     image_features = outputs.vision_hidden_states[-1]

In [ ]:
inputs.keys()

dict_keys(['pixel_values', 'image_grid_thw'])

In [ ]:
with torch.no_grad():
  out=model.visual(inputs["pixel_values"],inputs["image_grid_thw"])

In [ ]:
import torch.nn.functional as F

logits = model.lm_head(out)
predicted_tokens = torch.argmax(logits, dim=-1)
predicted_tokens.shape

torch.Size([3577])

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
decoded_text = tokenizer.decode(predicted_tokens)

decoded_text

'侃\tStringBuffer\tInputStream createStackNavigator临-Col\tstrncpy\tstrncpy\tstrncpy\tstrncpy\tstrncpy\tstrncpy\tfflush\tstrncpy\tstrncpy\tstrncpy\tstrncpy\tstrncpy\tstrncpy\tstrncpy\tStringBuffer\tstrncpy\tStringBuffer\tstrncpy\tStringBuffer\tStringBuffer CircularProgress\tstrncpy\tstrncpy\tstrncpy\tstrncpy\tStringBuffer\tStringBuffer\tstrncpy\tInputStream\tfflush CircularProgress CircularProgress CircularProgress CircularProgress CircularProgress CircularProgress CircularProgress.AutoComplete.ServletException.ServletExceptionoplanбал.ServletExceptionметacheracher CircularProgress},${},${enie ICollection},${\tfflush\tstrncpy\tstrncpy gratuites\tInputStream ApplicationException事业单位 withRouter withRouter CircularProgressiples❥ưaacbacbtoContain\tstrncpy\tstrncpy\tstrncpy\tstrncpy\tstrncpy\tstrncpy\tstrncpy\tstrncpystrcasecmp\tstrncpy\tstrncpy\tstrncpy\tstrncpy\tstrncpy\tstrncpy\tstrncpy\tfflush\tstrncpy%;"> CircularProgress\tstrncpy CircularProgress\tstrncpy CircularProgress\tfflush Circul

In [ ]:

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
            },
        ],
    }
]

processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Get the embedding layer
embedding_layer = model.model.embed_tokens

# Forward pass until the embedding layer
with torch.no_grad():
  embedding_output = embedding_layer(inputs.input_ids)


print("Embedding output shape:", embedding_output.shape)

Embedding output shape: torch.Size([1, 3598, 1536])


In [ ]:
inputs["input_ids"].shape

torch.Size([1, 3598])

In [ ]:
model.visual(inputs["pixel_values"])

TypeError: Qwen2VisionTransformerPretrainedModel.forward() missing 1 required positional argument: 'grid_thw'

In [ ]:
model

Qwen2VLForConditionalGeneration(
  (visual): Qwen2VisionTransformerPretrainedModel(
    (patch_embed): PatchEmbed(
      (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
    )
    (rotary_pos_emb): VisionRotaryEmbedding()
    (blocks): ModuleList(
      (0-31): 32 x Qwen2VLVisionBlock(
        (norm1): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
        (norm2): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
        (attn): VisionSdpaAttention(
          (qkv): Linear(in_features=1280, out_features=3840, bias=True)
          (proj): Linear(in_features=1280, out_features=1280, bias=True)
        )
        (mlp): VisionMlp(
          (fc1): Linear(in_features=1280, out_features=5120, bias=True)
          (act): QuickGELUActivation()
          (fc2): Linear(in_features=5120, out_features=1280, bias=True)
        )
      )
    )
    (merger): PatchMerger(
      (ln_q): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
      (mlp): Seq

In [ ]:
logits = model.lm_head(embedding_output)
print(logits.shape)

torch.Size([1, 3602, 151936])


In [ ]:
# default processer
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

# The default range for the number of visual tokens per image in the model is 4-16384. You can set min_pixels and max_pixels according to your needs, such as a token count range of 256-1280, to balance speed and memory usage.
# min_pixels = 256*28*28
# max_pixels = 1280*28*28
# processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct", min_pixels=min_pixels, max_pixels=max_pixels)

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
            },
            {"type": "text", "text": "Describe this image."},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

["The image depicts a serene beach scene with a woman and a dog. The woman is sitting on the sand, wearing a plaid shirt and black pants, and appears to be smiling. She is holding the dog's paw in a high-five gesture. The dog, which is a large breed, is sitting on the sand with its front paws raised, possibly in response to the woman's gesture. The background shows the ocean with gentle waves, and the sky is clear with a soft light, suggesting it might be either sunrise or sunset. The overall atmosphere is peaceful and joyful."]
